# SwissPhenoCam — Paper figures and tables

All inputs are read from the SwissPhenoCam release folder (`DATA_ROOT` in the setup cell).

In [ ]:
# §0 — Setup
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pathlib import Path
from scipy.stats import linregress
from tqdm import tqdm
import re

fig_folder = Path("figures")
fig_folder.mkdir(exist_ok=True)

# ── Paths ──────────────────────────────────────────────────────────────────
# All inputs come from the SwissPhenoCam release folder.
DATA_ROOT   = Path("/path/to/SwissPhenoCamDataset")  # <-- change this to your local path

RECORDS_CSV = DATA_ROOT / "phenology_records_curated.csv"
# Site table: ID, Location (site name), Elevation
ELEV_CSV    = DATA_ROOT / "auxiliary-data" / "site-info.csv"
# Timeseries root: location_sublocation / individual_id / year / full_name-GCC-{freq}-product.csv
TS_ROOT     = DATA_ROOT / "greenness-phenology"
# SPN species means per (species, year, elevation band).
SPN_CSV     = DATA_ROOT / "auxiliary-data" / "validation-data" / "validation_spn_means.csv"
# Expert visual annotations, one row per tree-year of the full set; *_mid = DOY midpoint of the annotated range
VISUAL_CSV  = DATA_ROOT / "auxiliary-data" / "validation-data" / "validation_visual_annotations.csv"

# ── Products / metrics ─────────────────────────────────────────────────────
PROD_3D = "GCC_3D_p90"
PROD_1D = "GCC_1D_p90"
PHASES  = ["GU_10","GU_25","GU_50","GU_75","GU_90","PEAK","GD_90","GD_75","GD_50","GD_25","GD_10"]

# Species sets, by number of tree-years in the v1.1 curated set.
# Main paper: the 6 most frequent species.
TOP6 = [
    'Larix decidua', 'Fagus sylvatica', 'Betula pendula',
    'Picea abies', 'Acer pseudoplatanus', 'Fraxinus excelsior',
]
# Supplementary material: the next 8, so that TOP14 holds every species with >= 50 tree-years
# (ties ordered by name: Aesculus and Corylus both have 51).
SUPP8 = [
    'Prunus avium', 'Pinus sylvestris', 'Malus domestica', 'Quercus robur',
    'Carpinus betulus', 'Aesculus hippocastanum', 'Corylus avellana', 'Juglans regia',
]
TOP14 = TOP6 + SUPP8
MIN_TREE_YEARS = 50  # species shown in the bar chart
# File-name suffix -> species set; LaTeX labels use '-supp' for the supplementary set.
SPECIES_SETS = {'': TOP6, '_supp': SUPP8}

# ── Load records ───────────────────────────────────────────────────────────
data = pd.read_csv(RECORDS_CSV)
print(f"Loaded {len(data)} tree-years, {data.columns.nunique()} columns")

# ── Add elevation ──────────────────────────────────────────────────────────
elev = pd.read_csv(ELEV_CSV, dtype={'ID': str})
elev_map = dict(zip(elev['ID'].astype(str), elev['Elevation']))
data['elevation'] = data['location'].astype(str).map(elev_map)

def _elev_band(e):
    if pd.isna(e): return np.nan
    if e < 600:   return 'Low'
    if e < 1000:  return 'Mid'
    if e < 1500:  return 'High'
    return 'Very High'

data['elevation_band'] = data['elevation'].apply(_elev_band)

# ── Add site name ─────────────────────────────────────────────────────────
_name_map = dict(zip(elev['ID'].astype(str), elev['Location']))
# Display names that differ from the camera-site table
SITE_NAMES = {
    '75d2c47bd00b9177189bd3f1fb6ff6ed': 'Villars Aiglon College',         # Chesieres - Aiglon
    '94abeb27e90adc9bb2f0b60834e4f8cf': 'Vallee du Trient',               # Trient Vallee
    '20b05e3c142bf4bbb6a6bf3be03545a8': 'Vallee du Trient - Marecottes',  # Trient Vallee - Marecottes
    '6a1352b9cd7062e5188cb951efcead15': 'Vallee du Trient - Vallorcine',  # Trient Vallee - Vallorcine
}
def _site_label(loc_id):
    if str(loc_id) in SITE_NAMES:
        return SITE_NAMES[str(loc_id)]
    raw = _name_map.get(str(loc_id), str(loc_id))
    return raw.split(":")[-1].strip()
data['site'] = data['location'].astype(str).map(_site_label)

# ── Derived columns ────────────────────────────────────────────────────────
data['image_type'] = data['location'].apply(
    lambda x: 'analysis' if str(x).isdigit() and len(str(x)) == 4 else 'panorama'
)
data['tree_year_id'] = data['full_name'].astype(str) + '-' + data['year'].astype(str)
data['year'] = data['year'].astype(int)

# ── Species colour map (shared across all plots) ───────────────────────────
tab20 = matplotlib.colormaps['tab20']
_order = [i for i in list(range(0, 20, 2)) + list(range(1, 20, 2)) if i not in (14, 15)]  # 14, 15 are grey
PALETTE = [tab20.colors[i] for i in _order]
OTHER_COLOR = (0.6, 0.6, 0.6)
_ty = data['species'].value_counts()
_ty = _ty.sort_index().sort_values(ascending=False, kind='stable')  # ties by name
COLOURED = list(_ty.index[:len(PALETTE)])
assert COLOURED[:len(TOP14)] == TOP14, "TOP14 is no longer the top of the tree-year ranking"
fixed_cmap = {s: PALETTE[i] for i, s in enumerate(COLOURED)}

def sp_color(species):
    return fixed_cmap.get(species, OTHER_COLOR)

def compact_name(name):
    parts = name.split()
    return f"{parts[0][0]}. {parts[-1]}" if len(parts) >= 2 else name

def sp_it(name):
    """Latin species name in italics for any matplotlib text (legend, label, title)."""
    return r"$\it{" + name.replace(" ", r"\ ") + "}$"

# Tables: style species index labels in italics (HTML; to_latex(convert_css=True) gives \itshape).
ITALIC_INDEX = lambda _: 'font-style: italic;'

print("Tree-years per species set:", {k or 'main': int(data['species'].isin(v).sum()) for k, v in SPECIES_SETS.items()})
print("elevation NA:", data['elevation'].isna().sum())
print("image_type counts:", data['image_type'].value_counts().to_dict())
data.head(2)

In [ ]:
# Export the species colour map: species -> hex colour, in colour-assignment order
# (the 18 species with the most tree-years). Any other species is drawn in OTHER_COLOR.
import json

species_colors = {s: matplotlib.colors.to_hex(c) for s, c in fixed_cmap.items()}
species_colors['Other species'] = matplotlib.colors.to_hex(OTHER_COLOR)
with open(fig_folder / 'species_colormap.json', 'w') as f:
    json.dump(species_colors, f, indent=2)
print(f"{len(fixed_cmap)} species colours written to {fig_folder / 'species_colormap.json'}")


## §1 — Species sample-size bar chart

In [ ]:
# Count tree-years per species; each row is one tree-year
ty_counts = (
    data.groupby('species')
    .agg(total_tree_years=('annotation_id', 'count'),
         n_trees=('annotation_id', 'nunique'))
    .reset_index()
)
ty_counts = ty_counts[ty_counts['total_tree_years'] >= MIN_TREE_YEARS].sort_values('total_tree_years', ascending=True)

colors = [sp_color(s) for s in ty_counts['species']]

fig, ax = plt.subplots(figsize=(3, len(ty_counts) * 0.15 + 0.5))
y_pos = range(len(ty_counts))
ax.barh(y_pos, ty_counts['total_tree_years'], color=colors)
ax.set_yticks(y_pos)
ax.set_yticklabels(ty_counts['species'], fontstyle='italic', fontsize=9)
for i, val in enumerate(ty_counts['total_tree_years']):
    ax.text(val * 1.01, i, str(int(val)), va='center', ha='left', fontsize=8)
ax.set_xlim(0, ty_counts['total_tree_years'].max() * 1.15)
ax.set_xlabel('Tree-years')
ax.set_xticks([0, 100, 200])
ax.xaxis.set_minor_locator(matplotlib.ticker.MultipleLocator(50))  # dotted line every 50 tree-years
ax.tick_params(axis='y', length=0)
ax.grid(axis='x', linestyle=':', linewidth=0.5, which='both')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
fig.savefig(fig_folder / 'species_tree_years.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart for all species, curated and full set together, one vertical bar per species.
# Filled bar = tree-years in the curated set; outlined bar behind it = tree-years in the full set
# (the curated set is a subset of the full set).
# Only species present in the curated set. Species are ordered by curated, then full count.
# Species without a fixed colour are grey; tree-years without a species label are not shown.
FULL_CSV = DATA_ROOT / "phenology_records_full.csv"
full_species = pd.read_csv(FULL_CSV, usecols=['species'])['species']
ty_all = (
    pd.concat([data['species'].value_counts().rename('curated'),
               full_species.value_counts().rename('full')], axis=1)
    .fillna(0).astype(int).rename_axis('species').reset_index()
    .query('curated > 0')
    .sort_values(['curated', 'full', 'species'], ascending=[False, False, True])
)

# 17 cm wide (full width of a two-column A4 paper), so it prints at 100 %; font sizes are printed sizes
fig, ax = plt.subplots(figsize=(17 / 2.54, 3.5))
x_pos = np.arange(len(ty_all))
colors = [sp_color(s) for s in ty_all['species']]
ax.bar(x_pos, ty_all['full'], width=0.8, facecolor='white', edgecolor=colors, linewidth=0.8)
ax.bar(x_pos, ty_all['curated'], width=0.8, color=colors)

ax.set_ylim(0, ty_all['full'].max() * 1.04)
ax.yaxis.set_major_locator(matplotlib.ticker.MultipleLocator(100))
ax.yaxis.set_minor_locator(matplotlib.ticker.MultipleLocator(50))  # dotted line every 50 tree-years
ax.grid(axis='y', linestyle=':', linewidth=0.9, which='major')  # every 100: slightly thicker
ax.grid(axis='y', linestyle=':', linewidth=0.5, which='minor')  # every 50
ax.set_axisbelow(True)
ax.set_ylabel('Tree-years', fontsize=7.5)
ax.tick_params(axis='y', labelsize=7)
ax.set_xticks(x_pos)
ax.set_xticklabels(ty_all['species'], fontstyle='italic', fontsize=6, rotation=60, ha='right',
                   rotation_mode='anchor')
ax.set_xlim(-0.7, len(ty_all) - 0.3)
ax.tick_params(axis='x', length=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(handles=[Patch(facecolor='grey', label='Curated set'),
                   Patch(facecolor='white', edgecolor='grey', label='Full set')],
          loc='upper right', fontsize=6.5, frameon=True, facecolor='white', edgecolor='none', framealpha=1)
plt.tight_layout()
fig.savefig(fig_folder / 'species_tree_years_all.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()
print(f"{len(ty_all)} species; "
      f"tree-years with a species label: curated {int(ty_all['curated'].sum())}, full {int(ty_all['full'].sum())}. "
      f"Not shown (no species label): curated {int(data['species'].isna().sum())}, full {int(full_species.isna().sum())}")


## §2b — Species presence across sites and years

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

# ── Build site × year presence matrix (all species) ──────────────────────
site_start = data.groupby('site')['year'].min().sort_values()
sorted_sites = site_start.index.tolist()
all_years = sorted(data['year'].unique())

# Global presence: 1 if ANY tree-year exists at this site-year, NaN otherwise
global_piv = (
    data.groupby(['year', 'site'])['annotation_id']
    .count().unstack(fill_value=0)
    .reindex(index=all_years, columns=sorted_sites)
)
# Cells with 0 count → NaN (site not active that year)
global_mask = global_piv > 0
n_sites = len(sorted_sites)

# ── Plot: 2 columns, one row per 2 species, one figure per species set ────
def plot_presence(species_list, suffix):
    nrows = int(np.ceil(len(species_list) / 2))
    fig, axes = plt.subplots(nrows, 2, figsize=(max(9, n_sites * 0.40), 13 * nrows / 5),
                             sharex=True, sharey=True, constrained_layout=True)
    axes.flat[0].invert_yaxis()

    for idx, (species, ax) in enumerate(zip(species_list, axes.flat)):
        sp_col = sp_color(species)

        # Species presence pivot
        sp_piv = (
            data[data['species'] == species]
            .groupby(['year', 'site'])['annotation_id']
            .count().unstack(fill_value=0)
            .reindex(index=all_years, columns=sorted_sites, fill_value=0)
        )
        sp_present = sp_piv > 0

        # Build 3-state matrix: NaN = no data, 0 = absent, 1 = present
        matrix = sp_present.astype(float).copy()
        matrix[~global_mask] = np.nan  # site-year has no data at all

        cmap = ListedColormap([(0.82, 0.82, 0.82), sp_col])
        cmap.set_bad(color='white')

        ax.pcolormesh(
            np.arange(len(sorted_sites) + 1),
            np.arange(len(all_years) + 1),
            matrix.values,
            cmap=cmap, vmin=0, vmax=1,
            edgecolors='white', linewidth=0.5,
        )

        # Dark edge on "present" cells for emphasis
        for yi, yr in enumerate(all_years):
            for xi, site in enumerate(sorted_sites):
                val = matrix.loc[yr, site] if yr in matrix.index and site in matrix.columns else np.nan
                if val == 1.0:
                    ax.add_patch(Rectangle((xi + 0.05, yi + 0.05), 0.9, 0.9,
                                 linewidth=0.6, edgecolor=(0.25, 0.25, 0.25),
                                 facecolor='none', zorder=2))

        # Row sums (number of sites with presence) on the right
        for yi, yr in enumerate(all_years):
            row_sum = int(matrix.loc[yr].sum())
            if row_sum > 0:
                ax.text(n_sites + 0.3, yi + 0.5, str(row_sum), ha='left', va='center',
                        fontsize=9, fontweight='bold', color=[0.75 * v for v in sp_col[:3]])  # darker shade, readable for light colours

        ax.set_yticks(np.arange(len(all_years)) + 0.5)
        ax.set_yticklabels(all_years, fontsize=8)
        ax.text(
                    x=14, y=1, s=f"$\\it{{{species.split()[0]}}}$" + f" $\\it{{{species.split()[-1]}}}$", va='center', ha='left',
                    fontsize=12)
        ax.set_aspect('equal')  # square cells, whatever the number of sites / years
        ax.tick_params(length=0)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # Shared x-axis: labels only on bottom row
    for ax in axes[-1]:
        ax.set_xticks(np.arange(len(sorted_sites)) + 0.5)
        ax.set_xticklabels(sorted_sites, rotation=45, fontsize=8, ha='right')

    fig.savefig(fig_folder / f"species_presence_site_year{suffix}.png", dpi=200, bbox_inches='tight')
    fig.savefig(fig_folder / f"species_presence_site_year{suffix}.pdf", bbox_inches='tight')
    plt.show()

for suffix, species_list in SPECIES_SETS.items():
    plot_presence(species_list, suffix)

In [ ]:
# ── Tree-years per site-year heatmap ──────────────────────────────────────
piv_ty = (
    data.groupby(['year', 'site'])['annotation_id']
    .count().unstack(fill_value=0)
    .reindex(index=all_years, columns=sorted_sites)
)
piv_ty[~global_mask] = np.nan

n_sites = len(sorted_sites)
width = max(8, n_sites * 0.4)
width=8
fig, ax = plt.subplots(figsize=(width, 4), constrained_layout=True)

cmap_ty = plt.cm.YlGnBu.copy()
cmap_ty.set_bad(color='white')

im = ax.pcolormesh(
    np.arange(n_sites + 1),
    np.arange(len(all_years) + 1),
    piv_ty.values,
    cmap=cmap_ty,
    edgecolors='white', linewidth=0.5,
)
# Annotate each cell
for yi, yr in enumerate(all_years):
    for xi, site in enumerate(sorted_sites):
        val = piv_ty.loc[yr, site] if yr in piv_ty.index and site in piv_ty.columns else np.nan
        if not np.isnan(val):
            ax.text(xi + 0.5, yi + 0.5, f"{int(val)}", ha='center', va='center', fontsize=7,
                    color='white' if val > piv_ty.max().max() * 0.6 else 'black')

ax.set_xticks(np.arange(n_sites) + 0.5)
ax.set_xticklabels(sorted_sites, rotation=45, ha='right', fontsize=7)
ax.set_yticks(np.arange(len(all_years)) + 0.5)
ax.set_yticklabels(all_years, fontsize=7)
# Row totals on the right
for yi, yr in enumerate(all_years):
    row_total = int(piv_ty.loc[yr].sum())
    ax.text(n_sites + 0.3, yi + 0.5, str(row_total), ha='left', va='center', fontsize=6, fontweight='bold')

ax.invert_yaxis()
ax.set_aspect('equal')  # square cells, whatever the number of sites / years
ax.tick_params(length=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# colour bar in axes coordinates, so it keeps the matrix height (right of the row totals)
cb = fig.colorbar(im, cax=ax.inset_axes([1.08, 0, 0.02, 1]), label='Tree-years')

fig.savefig(fig_folder / "tree_years_per_site_year.png", dpi=200, bbox_inches='tight')
fig.savefig(fig_folder / "tree_years_per_site_year.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# ── Raw observations per site-year heatmap ────────────────────────────────
obs_counts = {}
for (loc, subloc, yr), grp in data.groupby(['location', 'sublocation', 'year']):
    tree_id = grp.iloc[0]['full_name'].split('_')[0]
    csv_path = TS_ROOT / f"{loc}_{subloc}" / tree_id / str(yr) / f"{tree_id}_{yr}_raw-cc-data.csv"
    if csv_path.exists():
        raw = pd.read_csv(csv_path, usecols=['datetime'])
        raw['date'] = raw['datetime'].str.split('_').str[0]
        avg_per_day = len(raw) / raw['date'].nunique()
        site = _site_label(loc)
        obs_counts[(yr, site)] = max(obs_counts.get((yr, site), 0), avg_per_day)

df_obs = pd.DataFrame([
    {'year': yr, 'site': site, 'avg_per_day': n}
    for (yr, site), n in obs_counts.items()
])
piv_obs = (
    df_obs.pivot_table(index='year', columns='site', values='avg_per_day', aggfunc='max')
    .reindex(index=all_years, columns=sorted_sites)
)
piv_obs[~global_mask] = np.nan

n_sites = len(sorted_sites)
width = max(10, n_sites * 0.4)
fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)

cmap_obs = plt.cm.YlGnBu.copy()
cmap_obs.set_bad(color='white')

im = ax.pcolormesh(
    np.arange(n_sites + 1),
    np.arange(len(all_years) + 1),
    piv_obs.values,
    cmap=cmap_obs,
    edgecolors='white', linewidth=0.5,
)

ax.set_xticks(np.arange(n_sites) + 0.5)
ax.set_xticklabels(sorted_sites, rotation=45, ha='right', fontsize=7)
ax.set_yticks(np.arange(len(all_years)) + 0.5)
ax.set_yticklabels(all_years, fontsize=7)
ax.invert_yaxis()
ax.set_aspect('equal')  # square cells, whatever the number of sites / years
ax.tick_params(length=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# colour bar in axes coordinates, so it keeps the matrix height
cb = fig.colorbar(im, cax=ax.inset_axes([1.02, 0, 0.02, 1]), label='Avg. images per day')

fig.savefig(fig_folder / "raw_observations_per_site_year.png", dpi=200, bbox_inches='tight')
fig.savefig(fig_folder / "raw_observations_per_site_year.pdf", bbox_inches='tight')
plt.show()

## §3 — Season length by species × elevation

In [ ]:
GU_COL = f"{PROD_3D}-GU_50"
GD_COL = f"{PROD_3D}-GD_25"

sub3 = data[data['species'].isin(TOP14) & data['elevation_band'].isin(['Low','Mid','High'])].copy()

stats3 = (
    sub3.groupby(['species', 'elevation_band'])
    .agg(mu_up=(GU_COL, 'mean'), sigma_up=(GU_COL, 'std'),
         mu_down=(GD_COL, 'mean'), sigma_down=(GD_COL, 'std'),
         N=('tree_year_id', 'nunique'))
    .reset_index()
)

elevation_order = ['High', 'Mid', 'Low']
elev_labels = {'Low': '< 600 m', 'Mid': '600–1000 m', 'High': '1000–1500 m'}
LABEL_X = 323     # x position (DOY) of the elevation-band labels
BAR_ALPHA = 0.8  # same tint for all bands: the order (High, Mid, Low from top) tells them apart

bar_height = 0.2
group_gap  = 0.1
n_elev = len(elevation_order)

def plot_season_length(species_list, suffix):
    fig, ax = plt.subplots(figsize=(12, 1.0 * len(species_list) * n_elev * bar_height + 2))
    ax.invert_yaxis()

    yticks, ytick_labels = [], []
    missing_labels = []
    # Band labels on every species only when some species lacks a band (then missing bands are
    # struck through); otherwise only on the first species.
    have = stats3[stats3['species'].isin(species_list)].groupby('species')['elevation_band'].nunique()
    label_all = (have.reindex(species_list, fill_value=0) < n_elev).any()

    for i, species in enumerate(species_list):
        group_top = i * (n_elev * bar_height + group_gap)
        color = sp_color(species)
        for j, band in enumerate(elevation_order):
            row = stats3[(stats3['species'] == species) & (stats3['elevation_band'] == band)]
            # Elevation-band label right of the bars (all species or only the first, see label_all);
            # struck through when the band has no data
            if label_all or i == 0:
                label = ax.text(LABEL_X, group_top + j * bar_height, elev_labels[band], va='center', ha='left',
                                fontsize=9, color='dimgrey' if not row.empty else 'darkgrey')
                if row.empty:
                    missing_labels.append(label)
            if row.empty:
                continue
            row = row.iloc[0]
            y = group_top + j * bar_height
            width = row['mu_down'] - row['mu_up']
            ax.barh(y=y, width=width, left=row['mu_up'], height=bar_height * 0.85,
                    color=color, alpha=BAR_ALPHA, edgecolor='white', linewidth=0.5, zorder=3-j)
            ax.errorbar(row['mu_up'],   y, xerr=row['sigma_up'],   fmt='.', color='grey',
                        capsize=2, ms=0, zorder=4, elinewidth=1)
            ax.errorbar(row['mu_down'], y, xerr=row['sigma_down'], fmt='.', color='grey',
                        capsize=2, ms=0, zorder=4, elinewidth=1)
            ax.text(210, y, f"{width:.0f} days", va='center', ha='center', fontsize=10, zorder=5,
                    bbox=dict(boxstyle='round,pad=0.18', facecolor='white', alpha=0.65, edgecolor=color))
        centre = group_top + (n_elev - 1) * bar_height / 2
        yticks.append(centre)
        ytick_labels.append(species)

    ax.set_yticks(yticks)
    ax.set_yticklabels(ytick_labels, fontstyle='italic', fontsize=11)
    ax.set_xticks(range(100, 330, 20))
    ax.set_xlabel('Day of year (DOY)')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='x', linestyle='-', linewidth=0.6)
    ax.text(120, -0.25, 'Green-up 50%', color='green', fontsize=10, ha='center',
            bbox=dict(boxstyle='round,pad=0.18', facecolor='white', alpha=0.65, edgecolor='green'))
    ax.text(300, -0.25, 'Green-down 25%', ha='center', fontsize=10, color='#d94701',
            bbox=dict(boxstyle='round,pad=0.18', facecolor='white', alpha=0.65, edgecolor='#d94701'))
    ax.set_xlim(right=345)
    # Strike through labels of missing bands (a line across each label's drawn extent)
    fig.canvas.draw()
    for label in missing_labels:
        bb = label.get_window_extent().transformed(ax.transData.inverted())
        y = label.get_position()[1]
        ax.plot([bb.x0, bb.x1], [y, y], color='darkgrey', linewidth=0.8, clip_on=False)
    plt.tight_layout()
    fig.savefig(fig_folder / f'season_length_by_elevation{suffix}.pdf', bbox_inches='tight')
    plt.show()

for suffix, species_list in SPECIES_SETS.items():
    plot_season_length(species_list, suffix)

### Table — Average GU₅₀ and GD₂₅ per species × elevation band

In [ ]:
_GU = f"{PROD_3D}-GU_50"
_GD = f"{PROD_3D}-GD_25"

elevation_order_full = ['Low', 'Mid', 'High', 'Very High']
elev_labels_full = {
    'Low':       'Low\n(<600 m)',
    'Mid':       'Mid\n(600–1000 m)',
    'High':      'High\n(1000–1500 m)',
    'Very High': 'Very High\n(>1500 m)',
}

sub_t = data[data['species'].isin(TOP14)].copy()

stats_t = (
    sub_t.groupby(['species', 'elevation_band'])
    .agg(
        mu_up=(_GU, 'mean'), sigma_up=(_GU, 'std'),
        mu_down=(_GD, 'mean'), sigma_down=(_GD, 'std'),
        N=('tree_year_id', 'nunique'),
    )
    .reset_index()
)

def _fmt(mu, sigma):
    return f"{mu:.1f} ± {sigma:.1f}" if not pd.isna(mu) else "—"

def gu_gd_table(species_list):
    rows = []
    for species in species_list:
        row = {}
        for band in elevation_order_full:
            r = stats_t[(stats_t['species'] == species) & (stats_t['elevation_band'] == band)]
            if r.empty:
                row[(band, 'GU₅₀')] = "—"
                row[(band, 'GD₂₅')] = "—"
                row[(band, 'n')]     = "—"
            else:
                r = r.iloc[0]
                row[(band, 'GU₅₀')] = _fmt(r['mu_up'],   r['sigma_up'])
                row[(band, 'GD₂₅')] = _fmt(r['mu_down'], r['sigma_down'])
                row[(band, 'n')]     = int(r['N'])
        rows.append(row)

    idx = pd.Index(species_list, name='Species')
    cols = pd.MultiIndex.from_tuples(
        [(elev_labels_full[b], m) for b in elevation_order_full for m in ['GU₅₀', 'GD₂₅', 'n']],
        names=['Elevation', 'Metric'],
    )
    flat_rows = [
        [row[(b, m)] for b in elevation_order_full for m in ['GU₅₀', 'GD₂₅', 'n']]
        for row in rows
    ]
    return pd.DataFrame(flat_rows, index=idx, columns=cols)

for suffix, species_list in SPECIES_SETS.items():
    print('Main paper' if not suffix else 'Supplementary')
    display(gu_gd_table(species_list).style.map_index(ITALIC_INDEX, axis=0))

In [ ]:
def _table_to_latex(stats_df, species_order, elev_order, label='tab:gu50-gd25'):
    elev_headers = {
        'Low':       r'Low ($<$600\,m)',
        'Mid':       r'Mid (600--1000\,m)',
        'High':      r'High (1000--1500\,m)',
        'Very High': r'Very High ($>$1500\,m)',
    }
    n_bands = len(elev_order)

    L = []
    L += [
        r'\begin{table}[]',
        r'\centering',
        r'\caption{Statistics of Green-Up 50\% (GU$_{50}$) and Green-Down 25\% (GD$_{25}$) '
        r'across all years per species and elevation band. '
        r'Values are mean DOY $\pm$ standard deviation; $n$ is the number of tree-years. '
        r'Note that only a few species are present in the highest elevations '
        r'and absent species are denoted with empty cells (-).}',
        r'\small',
        r'\setlength{\tabcolsep}{3pt}',
        r'\resizebox{\textwidth}{!}{%',
        r'\begin{tabular}{l ' + ' '.join(['llr'] * n_bands) + r'}',
        r'\tophline',
    ]

    # Elevation band header row
    band_cols = ' & '.join(
        r'\multicolumn{3}{c}{' + elev_headers[b] + r'}' for b in elev_order
    )
    L.append(r' & ' + band_cols + r' \\')

    # \cmidrule lines
    cmidrules = ' '.join(
        r'\cmidrule(lr){' + f'{2 + i*3}-{4 + i*3}' + r'}' for i in range(n_bands)
    )
    L.append(cmidrules)

    # Sub-header row
    sub_cols = ' & '.join([r'GU$_{50}$', r'GD$_{25}$', r'$n$'] * n_bands)
    L.append(r'Species & ' + sub_cols + r' \\')
    L.append(r'\middlehline')

    # Data rows
    for species in species_order:
        cells = [r'\textit{' + species + r'}']
        for band in elev_order:
            row = stats_df[(stats_df['species'] == species) & (stats_df['elevation_band'] == band)]
            if row.empty:
                cells += ['--', '--', '--']
            else:
                r = row.iloc[0]
                cells += [
                    f"${r['mu_up']:.1f} \\pm {r['sigma_up']:.1f}$",
                    f"${r['mu_down']:.1f} \\pm {r['sigma_down']:.1f}$",
                    str(int(r['N'])),
                ]
        L.append(' & '.join(cells) + r' \\')

    L += [
        r'\bottomhline',
        r'\end{tabular}%',
        r'}',
        r'\label{' + label + r'}',
        r'\end{table}',
    ]
    return '\n'.join(L)


for suffix, species_list in SPECIES_SETS.items():
    print(_table_to_latex(stats_t, species_list, elevation_order_full,
                          label='tab:gu50-gd25' + suffix.replace('_', '-')))
    print()

## §4 — Transition-date uncertainty (1D vs 3D, 95 % CI)

In [ ]:
# Build ordered list of std column pairs
phase_order = ['GU_10','GU_25','GU_50','GU_75','GU_90','PEAK','GD_90','GD_75','GD_50','GD_25','GD_10']

cols_3d = [f"{PROD_3D}-{p}-std" for p in phase_order]
cols_1d = [f"{PROD_1D}-{p}-std" for p in phase_order]

# Keep only cols present in data
cols_3d_ok = [c for c in cols_3d if c in data.columns]
cols_1d_ok = [c for c in cols_1d if c in data.columns]

# Build label list from available 3D columns
available_phases = [c.replace(f"{PROD_3D}-", '').replace('-std', '') for c in cols_3d_ok]
label_map = {p: p.replace('_', ' ') for p in available_phases}

mu_3d = data[cols_3d_ok].mean()
mu_1d = data[cols_1d_ok].mean() if cols_1d_ok else pd.Series(dtype=float)

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.invert_yaxis()
y_labels = [label_map[p] for p in available_phases]

if not mu_1d.empty:
    # align 1D to same phases
    mu_1d_vals = [mu_1d.get(f"{PROD_1D}-{p}-std", np.nan) for p in available_phases]
    ax.barh(y=y_labels, width=np.array(mu_1d_vals) * 1.96, color='lightsteelblue', alpha=1, label='1D product')

ax.barh(y=y_labels, width=mu_3d.values * 1.96, color='royalblue', label='3D product')
ax.set_xlabel(r'Average transition date 95% CI ($1.96\sigma$) in days')
ax.set_xticks(range(0, 19, 1))
ax.set_xticks([x+0.5 for x in range(0, 19, 1)], minor=True)

ax.grid(axis='x', linestyle='-', linewidth=0.5)
ax.grid(axis='x', linestyle='--', linewidth=0.25, which='minor')

ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend()
plt.tight_layout()
fig.savefig(fig_folder / '95CI_interval_per_phase.pdf', bbox_inches='tight')
plt.show()

## §5 — Stratified quality heatmap

Low-elevation tree-years only. Species rows: main-paper species, analysis cameras only.
Camera-type rows: the main-paper species that have both camera types.
Each value is the average over 100 balanced subsamples, each stratum cut to the size of the smallest stratum.

In [ ]:
# Balanced subsampling, repeated: every stratum is subsampled to the size of the smallest
# stratum (N = min stratum size), the group means are computed, and this is repeated N_DRAWS
# times with different seeds; the heatmap shows the average over the draws.
# Low elevation only, so elevation does not differ between groups.
# - Species rows: TOP6, analysis cameras only; strata = species.
# - Camera-type rows: the TOP6 species that have both camera types; strata = species x camera type.
N_DRAWS = 100

std_cols = [f"{PROD_3D}-{p}-std" for p in ['GU_10','GU_25','GU_50','GU_75','GU_90',
                                             'GD_90','GD_75','GD_50','GD_25','GD_10']]
std_cols_ok = [c for c in std_cols if c in data.columns]
metric_cols = [f"{PROD_3D}-snr"] + std_cols_ok

def balanced_means(df, strata_cols, group_col, n_draws=N_DRAWS, seed=42):
    """Mean of metric_cols per group_col, averaged over n_draws balanced subsamples."""
    counts = df.groupby(strata_cols).size()
    n = int(counts.min())
    print(f"strata {strata_cols}: sizes {counts.to_dict()} -> N = {n}, {n_draws} draws")
    draws = [
        df.groupby(strata_cols).sample(n=n, random_state=seed + d)
          .groupby(group_col)[metric_cols].mean()
        for d in range(n_draws)
    ]
    return pd.concat(draws).groupby(level=0).mean()

low = data[data['species'].isin(TOP6) & (data['elevation_band'] == 'Low')]

# Species rows: analysis cameras only
quality = balanced_means(low[low['image_type'] == 'analysis'], ['species'], 'species').round(2)
quality = quality.loc[[s for s in TOP6 if s in quality.index]]

# ×1.96 for CI columns
quality[std_cols_ok] *= 1.96

# Rename columns for readability
rename = {f"{PROD_3D}-snr": 'SNR'}
rename.update({c: c.replace(f"{PROD_3D}-", '').replace('-std', ' CI') for c in std_cols_ok})
quality_disp = quality.rename(columns=rename)


In [ ]:
# Camera-type rows: only species with both camera types, so species mix is the same in both groups
both_types = [s for s in TOP6 if low.loc[low['species'] == s, 'image_type'].nunique() == 2]
print('Species with both camera types:', both_types)
quality_by_type = balanced_means(low[low['species'].isin(both_types)],
                                 ['species', 'image_type'], 'image_type').round(2)
quality_by_type[std_cols_ok] *= 1.96
quality_by_type_disp = quality_by_type.rename(columns=rename)


In [ ]:
# ── Quality heatmap: species (top) and camera types (bottom) in one figure ──
# Same numbers as the two tables above.
# SNR and CI have separate colours; each colour bar says which end is better.
# - CI columns: one shared scale, light = narrow CI. Square-root scale, so differences among
#   the many small CIs stay visible next to the few large ones.
# - SNR column: its own scale in blues, dark = high SNR (better).
# - Cells are vector rectangles (not an image), so the white gaps stay exact at any zoom.
# - Text is white on dark cells. Wider gaps separate SNR | green-up | green-down, and species | camera types.
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle

ci_cols = [c for c in quality_disp.columns if c != 'SNR']
type_labels = {'analysis': 'Analysis cameras', 'panorama': 'Panorama cameras'}
heat_df = pd.concat([quality_disp, quality_by_type_disp.rename(index=type_labels)])
n_sp = len(quality_disp)
vals = heat_df.values.astype(float)
is_snr = np.array([c == 'SNR' for c in heat_df.columns])
n_gu = sum(c.startswith('GU') for c in heat_df.columns)

cmap_ci, cmap_snr = matplotlib.colormaps['YlOrRd'], matplotlib.colormaps['Blues']
norm_ci = mcolors.PowerNorm(0.5, vmin=0, vmax=np.nanmax(vals[:, ~is_snr]))
norm_snr = mcolors.Normalize(vmin=np.nanmin(vals[:, is_snr]) - 2, vmax=np.nanmax(vals[:, is_snr]) + 2)
rgba = np.where(is_snr[None, :, None], cmap_snr(norm_snr(vals)), cmap_ci(norm_ci(vals)))

# Sized for a full-width figure in a two-column A4 paper: the saved PDF is FIG_WIDTH_IN wide,
# so it is printed at 100 % and the font sizes below are the printed sizes.
FIG_WIDTH_IN = 17 / 2.54
fig, ax = plt.subplots(figsize=(FIG_WIDTH_IN, 0.27 * len(heat_df) + 0.9))

# Cell centres: extra space between column groups (SNR | green-up | green-down)
# and between row groups (species | camera types). Units: one cell = 1.
CELL_GAP, COL_GROUP_GAP, ROW_GROUP_GAP = 0.06, 0.25, 0.45
col_group = np.array([0] + [1 if c.startswith('GU') else 2 for c in ci_cols])
xpos = np.arange(vals.shape[1]) + COL_GROUP_GAP * col_group
ypos = np.arange(vals.shape[0]) + ROW_GROUP_GAP * (np.arange(vals.shape[0]) >= n_sp)

for r in range(vals.shape[0]):
    for c in range(vals.shape[1]):
        rgb = rgba[r, c, :3]
        ax.add_patch(Rectangle((xpos[c] - 0.5 + CELL_GAP / 2, ypos[r] - 0.5 + CELL_GAP / 2),
                               1 - CELL_GAP, 1 - CELL_GAP, facecolor=rgb, edgecolor='none'))
        dark = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2] < 0.5
        ax.text(xpos[c], ypos[r], f"{vals[r, c]:.1f}", ha='center', va='center', fontsize=8,
                fontfamily='serif', color='white' if dark else 'black')
ax.set_xlim(xpos[0] - 0.5, xpos[-1] + 0.5)
ax.set_ylim(ypos[-1] + 0.5, ypos[0] - 0.5)   # first row at the top

col_labels = ['SNR\n(dB)'] + [re.sub(r'(GU|GD)_(\d+) CI', r'\1$_{\2}$', c) for c in ci_cols]
ax.set_xticks(xpos)
ax.set_xticklabels(col_labels, fontsize=7.5)
ax.xaxis.tick_top()
ax.set_yticks(ypos)
ax.set_yticklabels(heat_df.index, fontsize=7.5)
for lab in ax.get_yticklabels()[:n_sp]:
    lab.set_fontstyle('italic')
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)

# Group headers above the column labels
for x0, x1, text in [(xpos[1], xpos[n_gu], 'Green-up: 95% CI width (days)'),
                     (xpos[n_gu + 1], xpos[-1], 'Green-down: 95% CI width (days)')]:
    # x in data units, y as a fraction of the axes height (1 = top of the heatmap)
    ax.text((x0 + x1) / 2, 1.17, text, transform=ax.get_xaxis_transform(), ha='center', va='bottom', fontsize=7.5)
    ax.plot([x0 - 0.4, x1 + 0.4], [1.16, 1.16], transform=ax.get_xaxis_transform(),
            color='dimgrey', linewidth=0.8, clip_on=False)

# Colour bars right of the heatmap, side by side: CI (shared) and SNR, each with its better/worse end
for x, mappable, label, top, bottom in [
        (1.02, ScalarMappable(norm=norm_ci, cmap=cmap_ci), '95% CI width (days)', 'worse', 'better'),
        (1.115, ScalarMappable(norm=norm_snr, cmap=cmap_snr), 'SNR (dB)', 'better', 'worse')]:
    cb = fig.colorbar(mappable, cax=ax.inset_axes([x, 0, 0.015, 1]))
    cb.set_label(label, fontsize=7, labelpad=2)
    cb.ax.tick_params(labelsize=6.5, length=2, pad=1)
    cb.outline.set_visible(False)
    cb.ax.text(0.5, 1.02, top, transform=cb.ax.transAxes, ha='center', va='bottom', fontsize=6.5)
    cb.ax.text(0.5, -0.02, bottom, transform=cb.ax.transAxes, ha='center', va='top', fontsize=6.5)

# Labels and colour bars take a fixed width, so shrink the figure until the cropped PDF
# (bbox_inches='tight') is FIG_WIDTH_IN wide.
for _ in range(5):
    fig.canvas.draw()
    tight_w = fig.get_tightbbox(fig.canvas.get_renderer()).width
    w, h = fig.get_size_inches()
    fig.set_size_inches(w + FIG_WIDTH_IN - tight_w, h)
fig.savefig(fig_folder / 'quality_heatmap.pdf', bbox_inches='tight', pad_inches=0.02)
plt.show()


## §6 — Average temporal profiles

In [ ]:
def format_index(df):
    df.index = pd.to_datetime(df.index, format='%Y-%m-%d')
    df.index = df.index.dayofyear
    return df

def minmax_scale(s):
    mi, ma = s.min(), s.max()
    return (s - mi) / (ma - mi) if (ma - mi) > 0 else s - mi

def load_profiles(sub_df, root=TS_ROOT, freq='3D', agg='p90'):
    """Return dict {tree_year_id: DOY-indexed Series (min-max normalised)}.

    Reads the smoothed time series (*-GCC-{freq}-smoothed.csv) rather than the
    aggregated product, so curves show the actual fitted smooth.

    Path pattern: root / {location}_{sublocation} / {individual_id} / {year}
                       / {full_name}-GCC-{freq}-smoothed.csv
    where individual_id = full_name without the trailing _{year}.
    """
    col_candidates = [f'GCC-{freq}-{agg}_model', f'GCC-{freq}-{agg}', agg]
    profiles = {}
    for _, row in sub_df.iterrows():
        full_name = row['full_name']
        individual_id = full_name.rsplit('_', 1)[0]
        path = (root
                / f"{row['location']}_{row['sublocation']}"
                / individual_id
                / str(row['year'])
                / f"{full_name}-GCC-{freq}-smoothed.csv")
        if not path.exists():
            continue
        try:
            df = pd.read_csv(path, index_col=0)
            col = next((c for c in col_candidates if c in df.columns), None)
            if col is None:
                continue
            series = format_index(df[col].dropna())
            series = minmax_scale(series)
            profiles[row['tree_year_id']] = series
        except Exception:
            pass
    return profiles

def _filter_complete(profiles, sub_df, prod, phases_list):
    """Keep only tree-years present in profiles AND with all phase dates available."""
    phase_cols = [f"{prod}-{m}" for m in phases_list if f"{prod}-{m}" in sub_df.columns]
    mask = (
        sub_df['tree_year_id'].isin(profiles.keys()) &
        sub_df[phase_cols].notna().all(axis=1)
    )
    sub_filt = sub_df[mask]
    prof_filt = {k: v for k, v in profiles.items() if k in sub_filt['tree_year_id'].values}
    return prof_filt, sub_filt

def _add_phase_markers(ax, sub_df, prod, phases_list):
    _GU_COL = '#1b4332'
    _GD_COL = '#9b2226'
    _PK_COL = '#1b4332'
    for m in phases_list:
        col = f"{prod}-{m}"
        if col not in sub_df.columns:
            continue
        vals = sub_df[col].dropna()
        if vals.empty:
            continue
        median = vals.median()
        mean = vals.mean()
        std  = vals.std()
        h    = float(m.split('_')[1]) / 100 if '_' in m else 1.0
        up   = 'GU' in m
        color = _PK_COL if m == 'PEAK' else (_GU_COL if up else _GD_COL)
        ax.plot([mean - std, mean + std], [h, h],
                color=color, alpha=0.85, linewidth=2, solid_capstyle='round')
        ax.plot(median, h, 'o', color=color, markersize=5, alpha=0.95,
                markeredgecolor='white', markeredgewidth=0.8, zorder=5)
        label = 'PEAK' if m == 'PEAK' else m.split('_')[1]
        if m == 'PEAK' or m.endswith('90'):
            x  = mean - std - 2 if up else mean + std + 2
            ha = 'right' if up else 'left'
        elif m.endswith('10'):
            x  = mean + std + 2 if up else mean - std - 2
            ha = 'left' if up else 'right'
        else:
            x  = mean + std + 2
            ha = 'left'
        ax.text(x, h, label, ha=ha, va='center', size=7,
                color=color, alpha=0.9, fontweight='bold')

def _style_profile_ax(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_yticks([0, 0.5, 1])
    ax.set_xticks(range(0, 400, 50))
    ax.set_xticks(range(25, 400, 50), minor=True)
    ax.grid(axis='x', linestyle='-', linewidth=0.5)
    ax.grid(axis='x', linestyle=':', linewidth=0.5, which='minor')
    ax.set_axisbelow(True)
    ax.set_ylabel('GCC (normalised)')
    ax.set_xlabel('Day of year')

def plot_profiles_overlay(profiles, sub_df, species, ax=None, prod=PROD_3D):
    """Overlay all individual normalised smoothed GCC curves (alpha 0.1) + phase markers."""
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(12, 2.5))
    prof_filt, sub_filt = _filter_complete(profiles, sub_df, prod, PHASES)
    ax.set_facecolor('seashell')
    for series in prof_filt.values():
        ax.plot(series.index, series.values, color='green', alpha=0.1)
    _add_phase_markers(ax, sub_filt, prod, PHASES)
    _style_profile_ax(ax)
    ax.text(365//2, 0.05, species, ha='center', style='italic',
            bbox=dict(facecolor=fixed_cmap.get(species, 'white'), alpha=0.6))
    if own_fig:
        plt.tight_layout()
        return fig, ax

def plot_profiles_quantile_band(profiles, sub_df, species, ax=None, prod=PROD_3D,
                                quantiles=((0.05, 0.95), (0.10, 0.90), (0.25, 0.75))):
    """Shaded quantile bands + median curve over smoothed series."""
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(12, 2.5))
    prof_filt, sub_filt = _filter_complete(profiles, sub_df, prod, PHASES)
    ax.set_facecolor('seashell')
    all_doy = list(range(1, 367))
    
    mat = pd.concat(prof_filt.values(), axis=1).reindex(all_doy).interpolate(method='index', limit_area='inside').values  # (366, n)
    alphas = np.linspace(0.15, 0.35, len(quantiles))
    for (lo_q, hi_q), alpha in zip(quantiles, alphas):
        lo = np.nanpercentile(mat, lo_q * 100, axis=1)
        hi = np.nanpercentile(mat, hi_q * 100, axis=1)
        ax.fill_between(all_doy, lo, hi, color='green', alpha=alpha,
                        label=f'p{int(lo_q*100)}-p{int(hi_q*100)}')
    median = np.nanmedian(mat, axis=1)
    ax.plot(all_doy, median, color='darkgreen', linewidth=2, label='median')
    _add_phase_markers(ax, sub_filt, prod, PHASES)
    _style_profile_ax(ax)
    ax.legend(fontsize=8, loc='upper right')
    ax.text(-10, 0.9, f'{species} \n(n={len(prof_filt)})', ha='left', size=12, style='italic',
            bbox=dict(facecolor=fixed_cmap.get(species, 'white'), alpha=1, edgecolor='black', linewidth=0.))
    if own_fig:
        plt.tight_layout()
        return fig, ax

print("Profile loader and plotters defined (using smoothed time series).")

In [ ]:
# Average profile per species, one PDF each (supplementary species: _supp suffix)
FILTER_ELEV = ['Low', 'Mid', 'High']
MIN_TY = 30

for suffix, species_list in SPECIES_SETS.items():
    for species in tqdm(species_list):
        sub_sp = data[
            (data['species'] == species) &
            data['elevation_band'].isin(FILTER_ELEV)
        ]
        if len(sub_sp) < MIN_TY:
            print(f"skipped {species}: {len(sub_sp)} tree-years")
            continue
        profs = load_profiles(sub_sp)
        if len(profs) < 5:
            print(f"skipped {species}: {len(profs)} profiles")
            continue
        fig, ax = plt.subplots(figsize=(6, 3))
        plot_profiles_quantile_band(profs, sub_sp, species, ax=ax)
        plt.tight_layout()
        fig.savefig(fig_folder / f'avg_profile-{species.replace(" ", "_")}{suffix}.pdf', bbox_inches='tight')
        plt.close(fig)

---
## §V — Validation against external datasets

In [ ]:
# §V0 — Validation setup
from scipy import stats as sc_stats
from sklearn.metrics import root_mean_squared_error
import matplotlib.ticker as mticker


_STAT_BBOX = dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="lightgray")

### §V1 — Comparison with the Swiss Phenology Network (SPN)

Species-mean phenology dates from the Swiss Phenology Network are aggregated into three elevation bands (Low < 600 m, Mid 600–1000 m, High 1000–1500 m) matching SwissPhenoCam's classification. SwissPhenoCam transition dates are averaged across all tree-years in each (species, year, elevation-band) stratum (minimum N=5) and correlated with the SPN means. All species with SPN data are used, not only the 14 species of the two sets. *Picea abies* is excluded as SPN covers needle emergence/colouring rather than broadleaf phenology.

In [ ]:
swm_all = (
    pd.read_csv(SPN_CSV)
    .set_index(['species', 'year', 'elevation_band'])
    .sort_index()[['spn_up', 'spn_down']]
)
print(f"SPN data loaded: {len(swm_all)} (species, year, elevation_band) entries")
print("SPN species:", sorted(swm_all.index.get_level_values('species').unique()))
swm_all.head()

In [ ]:
# Spring: GU thresholds vs SPN leaf unfolding
GU_PHASES_V  = ["GU_10","GU_25","GU_50","GU_75","GU_90"]
spn_phase_up = "spn_up"
min_N        = 5
ignore_sp    = ['Picea abies']
#ignore_sp = []

# All species with SPN data, except ignore_sp
spn_species = swm_all.index.get_level_values('species').unique()
sub_v = data[data.species.isin(spn_species) & ~data.species.isin(ignore_sp)]

metrics_gu, values_gu = [], {}
for ph in GU_PHASES_V:
    col = f"{PROD_3D}-{ph}"
    if col not in sub_v.columns:
        continue
    stats = (
        sub_v.groupby(['species','year','elevation_band'])
        .agg(doy_cam=(col, 'mean'), N=('full_name', 'nunique'))
        .reset_index()
    )
    stats = stats[stats.N > min_N]
    combined = pd.concat(
        [swm_all[spn_phase_up],
         stats.set_index(['species','year','elevation_band'])[['doy_cam','N']]],
        axis=1
    )
    combined = combined[
        (combined.N > 0) & combined[spn_phase_up].notna() & combined.doy_cam.notna()
    ]
    x, y = combined[spn_phase_up].values, combined.doy_cam.values
    _, _, r, _, _ = sc_stats.linregress(x, y)
    metrics_gu.append(dict(Phase=ph, n=len(x), R2=round(r**2, 3),
                           Bias=round(float(np.mean(y - x)), 2),
                           RMSE=round(root_mean_squared_error(x, y), 2)))
    values_gu[ph] = combined

pd.DataFrame(metrics_gu)

In [ ]:
gu_phase  = "GU_50"
combined  = values_gu[gu_phase]
valid     = combined.dropna()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
mi, ma = int(valid.iloc[:, :2].min().min()), int(valid.iloc[:, :2].max().max())
ax.plot([mi, ma], [mi, ma], color='grey', linestyle='--', alpha=.5)

ax.scatter(
    valid.doy_cam, valid[spn_phase_up],
    s=2*valid.N, alpha=.9, edgecolor='grey',
    color=[sp_color(s) for (s,y,h),_ in valid.iterrows()]
)

legend_handles = [
    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=sp_color(s), markersize=8, label=sp_it(s))
    for s in valid.index.get_level_values('species').unique()
]
n_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey',
           markersize=np.sqrt(2*n), label=f'N = {n}')
    for n in [10, 20, 50]
]
size_legend = ax.legend(handles=n_handles, title='Sample size', frameon=True,
                        fontsize=8, title_fontsize=8, loc='lower right')
ax.add_artist(size_legend)
ax.legend(handles=legend_handles, title='Species', frameon=True,
          fontsize=8, title_fontsize=8, loc='upper left')

ax.set_xlabel("SwissPhenoCam Green-Up 50% \n  mean (DOY)")
ax.set_ylabel("Swiss Phenology Network\nLeaf unfolding / Needle emergence 50% — mean (DOY)")

x_v, y_v = valid[spn_phase_up].values, valid.doy_cam.values
slope, intercept, r, *_ = linregress(x_v, y_v)
r2   = r ** 2
rmse = root_mean_squared_error(y_v, x_v)
bias = np.mean(x_v - y_v)

ax.text(0.97, 0.40, f"R² = {r2:.2f}\nbias = {bias:.1f} d\nRMSE = {rmse:.1f} d",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9, bbox=_STAT_BBOX)
ax.plot([mi, ma], [slope*mi+intercept, slope*ma+intercept], color='steelblue', linewidth=1.2)

ax.set_facecolor('seashell')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.grid(axis='x', linestyle='-', linewidth=0.5)
ax.grid(axis='x', linestyle=':', linewidth=0.5, which='minor')
ax.grid(axis='y', linestyle='-', linewidth=0.5)
ax.grid(axis='y', linestyle=':', linewidth=0.5, which='minor')

plt.tight_layout()
fig.savefig(fig_folder / 'SPN-validation-GU50.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Autumn: GD thresholds vs SPN leaf colouring
GD_PHASES_V  = ["GD_10","GD_25","GD_50","GD_75","GD_90"]
spn_phase_dn = "spn_down"

metrics_gd, values_gd = [], {}
for ph in GD_PHASES_V:
    col = f"{PROD_3D}-{ph}"
    if col not in sub_v.columns:
        continue
    stats = (
        sub_v.groupby(['species','year','elevation_band'])
        .agg(doy_cam=(col, 'mean'), N=('full_name', 'nunique'))
        .reset_index()
    )
    stats = stats[stats.N > min_N]
    combined = pd.concat(
        [swm_all[spn_phase_dn],
         stats.set_index(['species','year','elevation_band'])[['doy_cam','N']]],
        axis=1
    )
    combined = combined[
        (combined.N > 0) & combined[spn_phase_dn].notna() & combined.doy_cam.notna()
    ]
    x, y = combined[spn_phase_dn].values, combined.doy_cam.values
    _, _, r, _, _ = sc_stats.linregress(x, y)
    metrics_gd.append(dict(Phase=ph, n=len(x), R2=round(r**2, 3),
                           Bias=round(float(np.mean(y - x)), 2),
                           RMSE=round(root_mean_squared_error(x, y), 2)))
    values_gd[ph] = combined

pd.DataFrame(metrics_gd)

In [ ]:
gd_phase  = "GD_25"
combined  = values_gd[gd_phase]
valid     = combined.dropna()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
mi, ma = int(valid.iloc[:, :2].min().min()), int(valid.iloc[:, :2].max().max())
ax.plot([mi-5, ma+10], [mi-5, ma+10], color='grey', linestyle='--', alpha=.5)

ax.scatter(
    valid.doy_cam, valid[spn_phase_dn],
    s=2*valid.N, alpha=.9, edgecolor='grey',
    color=[sp_color(s) for (s,y,h),_ in valid.iterrows()]
)

legend_handles = [
    Line2D([0],[0], marker='o', color='w',
           markerfacecolor=sp_color(s), markersize=8, label=sp_it(s))
    for s in valid.index.get_level_values('species').unique()
]
n_handles = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='grey',
           markersize=np.sqrt(2*n), label=f'N = {n}')
    for n in [10, 20, 50]
]
size_legend = ax.legend(handles=n_handles, title='Sample size', frameon=True,
                        fontsize=8, title_fontsize=8, loc='lower right')
ax.add_artist(size_legend)
ax.legend(handles=legend_handles, title='Species', frameon=True,
          fontsize=8, title_fontsize=8, loc='upper left')

ax.set_xlabel("SwissPhenoCam Green-Down 25% \n mean (DOY)")
ax.set_ylabel("Swiss Phenology Network\nLeaf / Needle colouring 50% — mean (DOY)")

y_v, x_v = valid[spn_phase_dn].values, valid.doy_cam.values
slope, intercept, r, *_ = linregress(x_v, y_v)
r2   = r ** 2
rmse = root_mean_squared_error(y_v, x_v)
bias = np.mean(x_v - y_v)

ax.text(0.33, 0.40, f"R² = {r2:.2f}\nbias = {bias:.1f} d\nRMSE = {rmse:.1f} d",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=9, bbox=_STAT_BBOX)
ax.plot([mi, ma], [slope*mi+intercept, slope*ma+intercept], color='steelblue', linewidth=1.2)

ax.set_facecolor('seashell')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
ax.grid(axis='x', linestyle='-', linewidth=0.5)
ax.grid(axis='x', linestyle=':', linewidth=0.5, which='minor')
ax.grid(axis='y', linestyle='-', linewidth=0.5)
ax.grid(axis='y', linestyle=':', linewidth=0.5, which='minor')
ax.set_xlim(270, 320)
ax.set_ylim(270, 320)
plt.tight_layout()
fig.savefig(fig_folder / 'SPN-validation-GD25.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Combined SPN agreement table for the paper: green-up (left) | green-down (right).
# Bold = best value per panel and metric (highest R², bias closest to 0, lowest RMSE),
# compared after rounding to 2 decimals, so ties are all bold.
def _spn_panel(metrics, phases):
    df = pd.DataFrame(metrics).set_index('Phase').loc[phases, ['R2', 'Bias', 'RMSE']].round(2)
    score = {'R2': df['R2'], 'Bias': -df['Bias'].abs(), 'RMSE': -df['RMSE']}
    return {m: [rf"\textbf{{{v:.2f}}}" if s == score[m].max() else f"{v:.2f}"
                for v, s in zip(df[m], score[m])]
            for m in df.columns}

gu_phases = ["GU_10", "GU_25", "GU_50", "GU_75", "GU_90"]
gd_phases = ["GD_90", "GD_75", "GD_50", "GD_25", "GD_10"]
gu_panel = _spn_panel(metrics_gu, gu_phases)
gd_panel = _spn_panel(metrics_gd, gd_phases)

def _head(ph):
    kind, pct = ph.split('_')
    return rf"{kind} ${pct}\%$"

lines = [
    r"\begin{table}[]",
    r"    \caption{Agreement between different green-up (spring, left panel) and green-down (autumn, right panel) "
    r"transition dates and the MeteoSwiss SPN's mean values for leaf unfolding $50\%$ and leaf or needle "
    r"colouration $50\%$, respectively.}",
    r"    \centering",
    r"    \small",
    r"    \begin{tabular}{l|cccccc|ccccc}",
    r"        \tophline",
    "         & " + " & ".join(_head(p) for p in gu_phases) + " & & "
    + " & ".join(_head(p) for p in gd_phases) + r" \\",
    r"        \middlehline",
]
for metric, label in [('R2', r"R$^2$"), ('Bias', "Bias"), ('RMSE', "RMSE")]:
    lines.append(f"        {label} & " + " & ".join(gu_panel[metric]) + " & & "
                 + " & ".join(gd_panel[metric]) + r" \\")
lines += [
    r"        \bottomhline",
    r"    \end{tabular}",
    r"    \label{tab:gu_gd_combined}",
    r"\end{table}",
]
print("\n".join(lines))

#### *Picea abies* alone (supplementary)

Same spring comparison for *Picea abies* only, which is excluded above (the SPN file has no colouring dates for *Picea*).

In [ ]:
# Spring: GU thresholds and PEAK vs SPN needle emergence, Picea abies only (same min_N as above).
# LaTeX table: phases as columns, metrics as rows (same layout as the combined SPN table).
pic = data[data.species == 'Picea abies']
spn_pic = swm_all.loc[['Picea abies'], spn_phase_up]

PIC_PHASES = GU_PHASES_V + ['PEAK']
metrics_pic = {}
for ph in PIC_PHASES:
    stats = (
        pic.groupby(['species', 'year', 'elevation_band'])
        .agg(doy_cam=(f"{PROD_3D}-{ph}", 'mean'), N=('full_name', 'nunique'))
    )
    combined = pd.concat([spn_pic, stats[stats.N > min_N]], axis=1).dropna()
    x, y = combined[spn_phase_up].values, combined.doy_cam.values
    metrics_pic[ph] = {'R2': sc_stats.linregress(x, y).rvalue ** 2,
                       'Bias': np.mean(y - x),
                       'RMSE': root_mean_squared_error(x, y)}

def _head(ph):
    if ph == 'PEAK':
        return 'PEAK'
    kind, pct = ph.split('_')
    return rf"{kind} ${pct}\%$"

lines = [
    r"\begin{table}[]",
    r"    \caption{Agreement between the green-up transition dates and peak date of \textit{Picea abies} and the MeteoSwiss "
    rf"SPN's mean values for needle emergence $50\%$ (n = {len(x)} year and elevation-band groups). "
    r"Bias and RMSE are in days.}",
    r"    \centering",
    r"    \small",
    r"    \begin{tabular}{l|" + "c" * len(PIC_PHASES) + "}",
    r"        \tophline",
    "         & " + " & ".join(_head(p) for p in PIC_PHASES) + r" \\",
    r"        \middlehline",
]
for key, label, fmt in [('R2', r"R$^2$", "{:.2f}"), ('Bias', "Bias", "${:+.1f}$"), ('RMSE', "RMSE", "{:.1f}")]:
    lines.append(f"        {label} & " + " & ".join(fmt.format(metrics_pic[p][key]) for p in PIC_PHASES) + r" \\")
lines += [
    r"        \bottomhline",
    r"    \end{tabular}",
    r"    \label{tab:gu_spn_picea}",
    r"\end{table}",
]
print("\n".join(lines))


### §V2 — Comparison with expert visual annotations

Individual tree-year greenness transition dates are matched against phenophases hand-annotated on raw camera images by a trained expert. Annotations cover spring leaf unfolding (LUG, LUH, LUD, LUM) and autumn senescence (SCS, SCH, SCE); the mid-point of each date range is used as the reference observation.


In [ ]:
# Load expert visual annotations and join to current records
va = pd.read_csv(VISUAL_CSV)
joined = data.merge(va, on='full_name', how='inner')
print(f"Joined records: {len(joined)} tree-years")

print("\nAnnotation coverage (non-null rows per phase):")
for ph in ['LUG','LUH','LUD','LUM','SCS','SCH','SCE']:
    n = joined[f"{ph}_mid"].notna().sum() if f"{ph}_mid" in joined.columns else 0
    print(f"  {ph}: {n}")

In [ ]:
# Spring validation: GU thresholds vs leaf unfolding phases
LU_PHASES = ['LUG','LUH','LUD','LUM']
GU_THRESH = ['GU_10','GU_25','GU_50','GU_75','GU_90']

lu_mid_cols  = [f"{p}_mid" for p in LU_PHASES]
gu_full_cols = [f"{PROD_3D}-{t}" for t in GU_THRESH]
gu_ok = [c for c in gu_full_cols if c in joined.columns]

spring = joined.dropna(subset=lu_mid_cols + gu_ok)
print(f"Spring complete rows: {len(spring)}")

# Metrics for all (GU threshold × LU phase) pairs
rows_sp = []
for th in GU_THRESH:
    col = f"{PROD_3D}-{th}"
    if col not in spring.columns: continue
    for ph in LU_PHASES:
        mid_col = f"{ph}_mid"
        sub = spring[[mid_col, col]].dropna()
        if len(sub) < 10: continue
        x, y = sub[mid_col].values, sub[col].values
        _, _, r, _, _ = sc_stats.linregress(x, y)
        rows_sp.append(dict(GU=th, LU=ph, n=len(sub),
                            R2=round(r**2, 3),
                            Bias=round(float(np.mean(y - x)), 2),
                            RMSE=round(root_mean_squared_error(x, y), 2)))

df_sp = pd.DataFrame(rows_sp)
df_sp


In [ ]:
# Autumn validation: GD thresholds vs senescence phases
SC_PHASES = ['SCS','SCH','SCE']
GD_THRESH = ['GD_10','GD_25','GD_50','GD_75','GD_90']

sc_mid_cols  = [f"{p}_mid" for p in SC_PHASES]
gd_full_cols = [f"{PROD_3D}-{t}" for t in GD_THRESH]
gd_ok = [c for c in gd_full_cols if c in joined.columns]

autumn = joined.dropna(subset=sc_mid_cols + gd_ok)
print(f"Autumn complete rows: {len(autumn)}")

rows_au = []
for th in GD_THRESH:
    col = f"{PROD_3D}-{th}"
    if col not in autumn.columns: continue
    for ph in SC_PHASES:
        mid_col = f"{ph}_mid"
        sub = autumn[[mid_col, col]].dropna()
        if len(sub) < 10: continue
        vals = sub[[mid_col, col]].values
        p2, p98 = np.percentile(vals, 2), np.percentile(vals, 98)
        sub = sub[(sub[mid_col] >= p2) & (sub[mid_col] <= p98) &
                  (sub[col] >= p2) & (sub[col] <= p98)]
        if len(sub) < 10: continue
        x, y = sub[mid_col].values, sub[col].values
        _, _, r, _, _ = sc_stats.linregress(x, y)
        rows_au.append(dict(GD=th, SC=ph, n=len(sub),
                            R2=round(r**2, 3),
                            Bias=round(float(np.mean(y - x)), 2),
                            RMSE=round(root_mean_squared_error(x, y), 2)))

df_au = pd.DataFrame(rows_au)
df_au


In [ ]:
# ── Combined scatter: GU₅₀ vs LUD | GU₂₅ vs LUG | GD₂₅ vs SCE ─────────────
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

panels = []

# --- Panel 1: GU_25 vs LUG ---
GU_COL_2 = f"{PROD_3D}-GU_25"
sub2 = spring[[GU_COL_2, 'LUG_mid', 'species']].dropna()
panels.append(dict(ax=axes[0], x=sub2[GU_COL_2].values, y=sub2['LUG_mid'].values,
                   species=sub2['species'], xlabel="GU₂₅ (DOY)", ylabel="LUG (DOY)",
                   label="(a)"))

# --- Panel 0: GU_50 vs LUD ---
GU_COL_1 = f"{PROD_3D}-GU_50"
sub1 = spring[[GU_COL_1, 'LUD_mid', 'species']].dropna()
panels.append(dict(ax=axes[1], x=sub1[GU_COL_1].values, y=sub1['LUD_mid'].values,
                   species=sub1['species'], xlabel="GU₅₀ (DOY)", ylabel="LUD (DOY)",
                   label="(b)"))



# --- Panel 2: GD_25 vs SCE (clipped) ---
GD_COL = f"{PROD_3D}-GD_25"
sub3 = autumn[[GD_COL, 'SCE_mid', 'species']].dropna()
p2  = np.percentile(sub3[['SCE_mid', GD_COL]].values, 2)
p98 = np.percentile(sub3[['SCE_mid', GD_COL]].values, 98)
sub3 = sub3[(sub3['SCE_mid'] >= p2) & (sub3['SCE_mid'] <= p98) &
            (sub3[GD_COL] >= p2) & (sub3[GD_COL] <= p98)]
panels.append(dict(ax=axes[2], x=sub3[GD_COL].values, y=sub3['SCE_mid'].values,
                   species=sub3['species'], xlabel="GD₂₅ (DOY)", ylabel="SCE (DOY)",
                   label="(c)"))

all_species = set()
for p in panels:
    all_species |= set(p['species'].unique())

for p in panels:
    ax = p['ax']
    x, y = p['x'], p['y']
    mi, ma = int(min(x.min(), y.min())), int(max(x.max(), y.max()))
    ax.plot([mi-5, ma+5], [mi-5, ma+5], color='grey', linestyle='--', alpha=.5)

    colors = [sp_color(s) for s in p['species']]
    ax.scatter(x, y, color=colors, alpha=.6, s=15, edgecolor='grey', linewidth=0.3)

    slope, intercept, r, *_ = linregress(x, y)
    r2   = r ** 2
    rmse = root_mean_squared_error(y, x)
    bias = np.mean(y - x)
    ax.text(0.97, 0.03, f"R²={r2:.2f}  bias={bias:.1f}d  RMSE={rmse:.1f}d  n={len(x)}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5, bbox=_STAT_BBOX)
    ax.plot([mi-5, ma+5], [slope*(mi-5)+intercept, slope*(ma+5)+intercept],
            color='steelblue', linewidth=1.2)

    ax.set_xlabel(p['xlabel'], fontsize=8)
    ax.set_ylabel(p['ylabel'], fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor('seashell')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.xaxis.set_minor_locator(mticker.MultipleLocator(5))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(5))
    ax.grid(axis='both', linestyle='-', linewidth=0.4)
    ax.grid(axis='both', linestyle=':', linewidth=0.3, which='minor')
    ax.text(0.03, 0.97, p['label'], transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='left')

# Shared legend below all panels
sp_sorted = sorted(all_species & set(fixed_cmap.keys()))
handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=fixed_cmap[s],
                  markersize=5, label=sp_it(s), linewidth=0) for s in sp_sorted]
if all_species - set(fixed_cmap.keys()):
    handles.append(Line2D([0],[0], marker='o', color='w', markerfacecolor=OTHER_COLOR,
                          markersize=5, label='Other species', linewidth=0))
fig.legend(handles=handles, loc='lower center', ncol=min(len(handles), 5),
           fontsize=6.5, frameon=True, title_fontsize=7,
           bbox_to_anchor=(0.5, -0.02), columnspacing=3, handletextpad=0.3)

fig.subplots_adjust(bottom=0.33, wspace=0.3)
fig.savefig(fig_folder / 'Visual-validation-scatter-combined.pdf', bbox_inches='tight')
fig.savefig(fig_folder / 'Visual-validation-scatter-combined.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# ── Combined RMSE table (Spring + Autumn) ────────────────────────────────────
piv_sp = df_sp.pivot(index='GU', columns='LU', values='RMSE')
piv_sp = piv_sp[['LUG', 'LUH','LUD','LUM']]
piv_au = df_au.pivot(index='GD', columns='SC', values='RMSE')
piv_au = piv_au[['SCS', 'SCH','SCE']]


gu_order = [f"GU_{t}" for t in [10,25,50,75,90] if f"GU_{t}" in piv_sp.index]
gd_order = [f"GD_{t}" for t in [90,75,50,25,10] if f"GD_{t}" in piv_au.index]
piv_sp = piv_sp.loc[[g for g in gu_order if g in piv_sp.index]]
piv_au = piv_au.loc[[g for g in gd_order if g in piv_au.index]]

def _fmt(v):
    return f"{v:.1f}" if pd.notna(v) else "—"

def _bold_min_row(row):
    vals = row.dropna()
    if vals.empty:
        return [_fmt(v) for v in row]
    best = vals.min()
    return [f"\\textbf{{{_fmt(v)}}}" if pd.notna(v) and v == best else _fmt(v) for v in row]

lines = []
lu_cols = list(piv_sp.columns)
sc_cols = list(piv_au.columns)
n_lu = len(lu_cols)
n_sc = len(sc_cols)

lines.append(r"\begin{table}[ht]")
lines.append(r"\centering")
lines.append(r"\caption{RMSE (days) between SwissPhenoCam transition dates and expert visual annotations. "
             r"Best value per threshold in \textbf{bold}.}")
lines.append(r"\label{tab:rmse-visual}")
col_spec = "l" + "c" * n_lu + "@{\\hskip 12pt}" + "l" + "c" * n_sc
lines.append(r"\begin{tabular}{" + col_spec + "}")
lines.append(r"\toprule")

header = (r" & " + " & ".join(lu_cols) +
          r" & & " + " & ".join(sc_cols) + r" \\")
lines.append(r"\multicolumn{" + str(1 + n_lu) + r"}{c}{\textit{Spring (green-up)}} & "
             r"\multicolumn{" + str(1 + n_sc) + r"}{c}{\textit{Autumn (senescence)}} \\")
lines.append(r"\cmidrule(r){1-" + str(1 + n_lu) + r"} \cmidrule(l){" +
             str(2 + n_lu) + "-" + str(1 + n_lu + 1 + n_sc) + "}")
lines.append(header)
lines.append(r"\midrule")

max_rows = max(len(piv_sp), len(piv_au))
for i in range(max_rows):
    parts = []
    if i < len(piv_sp):
        idx = piv_sp.index[i]
        label = idx.replace("_", r"$_{") + r"}$"
        vals = _bold_min_row(piv_sp.iloc[i])
        parts.append(label + " & " + " & ".join(vals))
    else:
        parts.append(" & ".join([""] * (1 + n_lu)))

    if i < len(piv_au):
        idx = piv_au.index[i]
        label = idx.replace("_", r"$_{") + r"}$"
        vals = _bold_min_row(piv_au.iloc[i])
        parts.append(label + " & " + " & ".join(vals))
    else:
        parts.append(" & ".join([""] * (1 + n_sc)))

    lines.append(" & ".join(parts) + r" \\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_str = "\n".join(lines)
print(latex_str)